In [1]:
#imports and client

from openai import OpenAI
import os
from dotenv import load_dotenv
import json

load_dotenv()

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [15]:
#First Python Tool

def add(a,b):
    return a+b

def sub(a,b):
    return a-b

def multiply(a,b):
    return a*b

def division(a,b):
    return a%b

In [16]:
#Testing
print(multiply(125,8))

1000


In [25]:
#Defining the tool == The moel needs a description of the fucntion and its outputs. We provide that description using JSON schemas
TOOLS=[
    {
    "type":"function",
    "name":"add",
    "description":"Add two numbers.",
    "parameters":{
        "type":"object",
        "properties":{
            "a":{
                "type":"number",
                "description":"The first number."
            },
            "b":{
                "type":"number",
                "description":"The second number."
            }
        },
        "required":["a","b"],
    "additionalProperties":False
    },
    "strict":True
    },
    {
    "type":"function",
    "name":"sub",
    "description":"Subtract two numbers.",
    "parameters":{
        "type":"object",
        "properties":{
            "a":{
                "type":"number",
                "description":"The first number."
            },
            "b":{
                "type":"number",
                "description":"The second number."
            }
        },
        "required":["a","b"],
    "additionalProperties":False
    },
    "strict":True
    },
    {
    "type":"function",
    "name":"multiply",
    "description":"Multiply two numbers.",
    "parameters":{
        "type":"object",
        "properties":{
            "a":{
                "type":"number",
                "description":"The first number."
            },
            "b":{
                "type":"number",
                "description":"The second number."
            }
        },
        "required":["a","b"],
    "additionalProperties":False
    },
    "strict":True
    },
    {
    "type":"function",
    "name":"division",
    "description":"Divide two numbers.",
    "parameters":{
        "type":"object",
        "properties":{
            "a":{
                "type":"number",
                "description":"The first number."
            },
            "b":{
                "type":"number",
                "description":"The second number."
            }
        },
        "required":["a","b"],
    "additionalProperties":False
    },
    "strict":True
    }
]

print("Available tools:", [tool["name"] for tool in TOOLS])

Available tools: ['add', 'sub', 'multiply', 'division']


In [29]:
#Sending the request with the tool

response=client.responses.create(
    model="gpt-4o-mini",
    input="What is 25 added by 3?",
    tools=TOOLS,
    tool_choice="required"
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"a":25,"b":3}', call_id='call_TzD6IgEHnEtIrZODZDUf5Lnb', name='add', type='function_call', id='fc_02405b6e606e0df9006ae84d295c9487d2919ec95aee4614cb', async_=None, caller=None, namespace=None, status='completed')]


In [35]:
for item in response.output:
    if item.type == "function_call":
        print("Function Name:",item.name)
        print("Arguments:",item.arguments)
        print("Call ID:",item.call_id)

Function Name: add
Arguments: {"a":25,"b":3}
Call ID: call_TzD6IgEHnEtIrZODZDUf5Lnb


In [36]:
#Run the requested tool

for item in response.output:
    if item.type=="function_call":
        if item.name=="add":
            arguments=json.loads(item.arguments)

            result=add(**arguments)
            print("Tool result:",result)

Tool result: 28


In [37]:
#Send the result back to LLM

tool_response=client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=response.id,

    input=[
        {
            "type":"function_call_output",
            "call_id":item.call_id,
            "output":str(result)
        }
    ]
)

print(tool_response.output_text)

25 added to 3 equals 28.
